# Kaggle · 01 · Preprocessing — Compile IR & Build Graphs

**Session type: CPU (no GPU needed) — does NOT consume your 30 GPU hrs/week.**

This notebook:
1. Installs LLVM 14 and PyTorch Geometric
2. Loads POJ-104 C/C++ source files
3. Compiles each program to LLVM IR at -O0
4. Parses each IR file into a `HeteroData` graph (.pt file)
5. Packages the graphs as a Kaggle dataset for the training notebooks

**Estimated runtime: 45–90 minutes on a Kaggle CPU session.**
Run this once. The output dataset is reused by all training notebooks.

---

## Setup instructions
1. Create a new Kaggle notebook → Kernel type: **Notebook** → Accelerator: **None (CPU)**
2. Under *Data* → *Add data* → search `poj104` and add the dataset
3. Copy-paste this notebook or upload the `.ipynb` file
4. Run all cells
5. After completion, go to *Output* → *New Dataset* → name it `compiler-opt-graphs`

---

## 0 · Install Dependencies

In [ ]:
import subprocess, sys

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"STDERR: {r.stderr[-500:]}")
    return r.returncode

print("Installing LLVM 14...")
run("apt-get update -qq && apt-get install -y -qq llvm-14 clang-14")
run("update-alternatives --install /usr/bin/clang clang /usr/bin/clang-14 100")
run("update-alternatives --install /usr/bin/opt   opt   /usr/bin/opt-14   100")

print("Installing PyTorch Geometric...")
import torch
TORCH_VER = torch.__version__.split('+')[0]
CUDA_VER  = 'cpu'  # CPU session
run(f"pip install -q torch_geometric")
run(f"pip install -q torch_scatter torch_sparse torch_cluster -f "
    f"https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_VER}.html")

print("Installing other deps...")
run("pip install -q tqdm pandas")

print("Verifying tools...")
for tool in ['clang', 'opt']:
    r = subprocess.run([tool, '--version'], capture_output=True, text=True)
    print(f"  {tool}: {r.stdout.splitlines()[0] if r.returncode==0 else 'NOT FOUND'}")

## 1 · Graph Builder (embedded)

In [ ]:
import os, re, subprocess, tempfile, shutil, time
from pathlib import Path
from typing import Dict, List, Optional
import torch
from torch_geometric.data import HeteroData
import numpy as np

OPCODES = [
    'alloca','load','store',
    'add','sub','mul','sdiv','udiv','srem','urem',
    'fadd','fsub','fmul','fdiv',
    'and','or','xor','shl','lshr','ashr',
    'icmp','fcmp','br','switch','ret','unreachable',
    'call','invoke','phi','select',
    'getelementptr','bitcast',
    'zext','sext','trunc','ptrtoint','inttoptr','fpext','fptrunc',
    'extractvalue','insertvalue','other',
]
OPCODE_TO_IDX = {op:i for i,op in enumerate(OPCODES)}
NUM_OPCODES   = len(OPCODES)

def _get_opcode_idx(line):
    tokens=line.split()
    if not tokens: return OPCODE_TO_IDX['other']
    if '=' in tokens:
        eq=tokens.index('=')
        if eq+1<len(tokens): return OPCODE_TO_IDX.get(tokens[eq+1].split('(')[0],OPCODE_TO_IDX['other'])
    return OPCODE_TO_IDX.get(tokens[0].split('(')[0],OPCODE_TO_IDX['other'])

def _empty():
    d=HeteroData()
    d['func'].x=torch.zeros((1,3)); d['bb'].x=torch.zeros((1,5)); d['instr'].x=torch.zeros((1,NUM_OPCODES))
    for et in [('bb','cfg','bb'),('instr','dfg','instr'),('instr','belongs','bb'),('func','calls','func')]:
        d[et].edge_index=torch.zeros((2,0),dtype=torch.long)
    return d

def compile_to_ir(src, out_dir):
    out=Path(out_dir)/f'{Path(src).stem}.ll'
    r=subprocess.run(['clang','-S','-emit-llvm','-O0','-Xclang','-disable-O0-optnone',
                      '-fno-discard-value-names','-w',str(src),'-o',str(out)],
                     capture_output=True,timeout=30)
    return str(out) if r.returncode==0 and out.exists() else None

def count_instructions(ll_path):
    skip={'','{','}'}; ct=0
    with open(ll_path,errors='replace') as f:
        for l in f:
            s=l.strip()
            if s and not any(s.startswith(p) for p in [';','!','define','declare','@','target','attributes','source_filename']) and s not in skip:
                ct+=1
    return max(ct,1)

def ir_to_graph(ir_text, edge_config=None):
    cfg={'use_cfg':True,'use_dfg':True,'use_belongs':True,'use_calls':True}
    if edge_config: cfg.update(edge_config)
    func_nodes,bb_nodes,instr_nodes=[],[],[]
    cfg_e,dfg_e,bel_e,call_e=[],[],[],[]
    fn2i,bb2i,reg2i={},{},{}
    pending=[]
    in_f=False; cfn=''; cfi=-1; cbn=None; cbi=-1; bd=0
    bni=bhb=bhc=bhp=bhr=0; gbb=0; gi=0

    def flush():
        nonlocal bni,bhb,bhc,bhp,bhr
        if cbn is not None and cbi>=0:
            while len(bb_nodes)<=cbi: bb_nodes.append(None)
            bb_nodes[cbi]=(cbn,cfi,[float(bni),float(bhb),float(bhc),float(bhp),float(bhr)])
        bni=bhb=bhc=bhp=bhr=0

    for raw in ir_text.splitlines():
        line=raw.strip()
        if not line or line.startswith(';') or line.startswith('!'): continue
        m=re.match(r'define\s+\S+\s+(@[\w.$]+)\s*\(([^)]*)\)',line)
        if m:
            in_f=True; cfn=m.group(1); na=len([a for a in m.group(2).split(',') if a.strip()])
            cfi=len(func_nodes); fn2i[cfn]=cfi; func_nodes.append((cfn,na))
            cbn=None; bd=0; continue
        if not in_f: continue
        bd+=line.count('{')-line.count('}')
        if bd<=0 and not line.startswith('define'):
            flush(); in_f=False; cbn=None; bd=0; continue
        bm=re.match(r'^([\w.$]+):\s*(?:;.*)?$',line)
        if bm and not any(line.startswith(p) for p in ['%','@','!']):
            flush(); cbn=bm.group(1); cbi=gbb; bb2i[f'{cfn}::{cbn}']=gbb; gbb+=1; continue
        if cbn is None and bd>0:
            cbn='entry'; cbi=gbb; bb2i[f'{cfn}::entry']=gbb; gbb+=1
        if cbn is None or any(line.startswith(p) for p in ['!','target','attributes']): continue
        oi=_get_opcode_idx(line)
        instr_nodes.append((line,oi)); ii=gi; gi+=1; bni+=1
        if cfg['use_belongs']: bel_e.append((ii,cbi))
        dm=re.match(r'(%[\w.$]+)\s*=',line)
        if dm: reg2i[f'{cfn}::{dm.group(1)}']=ii
        if cfg['use_dfg']:
            rhs=line.split('=',1)[-1] if '=' in line else line
            for u in re.findall(r'%[\w.$]+',rhs):
                uk=f'{cfn}::{u}'
                if uk in reg2i: dfg_e.append((reg2i[uk],ii))
        op=OPCODES[oi]
        if op in ('br','switch') and cfg['use_cfg']:
            bhb=1
            for t in re.findall(r'label\s+%?([\w.$]+)',line):
                tk=f'{cfn}::{t}'
                if tk in bb2i: cfg_e.append((cbi,bb2i[tk]))
                else: pending.append((cbi,tk))
        elif op in ('call','invoke'):
            bhc=1
            if cfg['use_calls']:
                cm=re.search(r'@([\w.$]+)\s*\(',line)
                if cm: call_e.append((cfi,'@'+cm.group(1)))
        elif op=='phi': bhp=1
        elif op=='ret': bhr=1

    for sb,tk in pending:
        if tk in bb2i: cfg_e.append((sb,bb2i[tk]))
    rcalls=[(ci,fn2i[cn]) for ci,cn in call_e if cn in fn2i]
    bb_nodes=[b for b in bb_nodes if b is not None]
    if not bb_nodes or not instr_nodes: return _empty()
    nf,nb,ni=len(func_nodes),len(bb_nodes),len(instr_nodes)
    fbc=[0]*nf; fic=[0]*nf
    for _,fi,ft in bb_nodes:
        if 0<=fi<nf: fbc[fi]+=1; fic[fi]+=int(ft[0])
    fx=torch.tensor([[float(fbc[i]),float(fic[i]),float(func_nodes[i][1])] for i in range(nf)],dtype=torch.float)
    bx=torch.tensor([ft for _,_,ft in bb_nodes],dtype=torch.float)
    ix=torch.zeros((ni,NUM_OPCODES))
    for i,(_,oi) in enumerate(instr_nodes): ix[i,oi]=1.0
    def mk(es):
        if es: s,d=zip(*es); return torch.tensor([list(s),list(d)],dtype=torch.long)
        return torch.zeros((2,0),dtype=torch.long)
    d=HeteroData()
    d['func'].x=fx; d['bb'].x=bx; d['instr'].x=ix
    d['bb','cfg','bb'].edge_index      =mk(cfg_e  if cfg['use_cfg']     else [])
    d['instr','dfg','instr'].edge_index=mk(dfg_e  if cfg['use_dfg']     else [])
    d['instr','belongs','bb'].edge_index=mk(bel_e if cfg['use_belongs'] else [])
    d['func','calls','func'].edge_index=mk(rcalls  if cfg['use_calls']  else [])
    return d

print(f"Graph builder ready. Opcode vocab: {NUM_OPCODES}")

## 2 · Locate POJ-104 Source Files

In [ ]:
from pathlib import Path
import random

# Standard Kaggle input paths to check for POJ-104
CANDIDATE_ROOTS = [
    Path('/kaggle/input/poj-104'),
    Path('/kaggle/input/poj104'),
    Path('/kaggle/input/poj-104-dataset'),
]

POJ_ROOT = next((p for p in CANDIDATE_ROOTS if p.exists()), None)

if POJ_ROOT is None:
    # If dataset name differs, find any directory with .c/.cpp files
    for p in Path('/kaggle/input').iterdir():
        c_files = list(p.rglob('*.c')) + list(p.rglob('*.cpp'))
        if len(c_files) > 50:
            POJ_ROOT = p
            break

if POJ_ROOT:
    all_sources = sorted(POJ_ROOT.rglob('*.c')) + sorted(POJ_ROOT.rglob('*.cpp'))
    print(f"POJ-104 root  : {POJ_ROOT}")
    print(f"Source files  : {len(all_sources)}")
    # Show directory structure
    problem_dirs = sorted(set(f.parent for f in all_sources))
    print(f"Problem dirs  : {len(problem_dirs)}")
    print(f"Example paths : {all_sources[0]}")
    print(f"              : {all_sources[-1]}")
else:
    print("POJ-104 not found. Using synthetic mini-dataset for demonstration.")
    # Create synthetic mini programs
    SYNTH_DIR = Path('/kaggle/working/synth_src')
    SYNTH_DIR.mkdir(exist_ok=True)
    templates = [
        '#include<stdio.h>\nint f(int n){return n<=1?n:f(n-1)+f(n-2);}\nint main(){printf("%d\\n",f(10));}',
        '#include<stdio.h>\nvoid bsort(int*a,int n){for(int i=0;i<n;i++)for(int j=i+1;j<n;j++)if(a[i]>a[j]){int t=a[i];a[i]=a[j];a[j]=t;}}\nint main(){int a[]={3,1,4,1,5};bsort(a,5);printf("%d\\n",a[0]);}',
        '#include<stdio.h>\nint gcd(int a,int b){return b?gcd(b,a%b):a;}\nint main(){printf("%d\\n",gcd(48,18));}',
    ]
    all_sources = []
    for i, t in enumerate(templates * 40):
        p = SYNTH_DIR / f'prog_{i:03d}.c'
        p.write_text(t)
        all_sources.append(p)
    print(f"Created {len(all_sources)} synthetic programs")

## 3 · Train/Val/Test Split

In [ ]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# POJ-104 is organised by problem (1–104). We split at the problem level
# to prevent any training leak.
# We take max 5 solutions per problem to keep compute manageable.

MAX_PER_PROBLEM = 5
N_TRAIN_PROBS   = 80
N_VAL_PROBS     = 12
N_TEST_PROBS    = 12

# Group sources by parent directory (= problem)
from collections import defaultdict
prob_to_files = defaultdict(list)
for f in all_sources:
    prob_to_files[f.parent.name].append(f)

problem_ids = sorted(prob_to_files.keys())
random.shuffle(problem_ids)

# If fewer than 104 problems, adjust proportionally
total = len(problem_ids)
if total < N_TRAIN_PROBS + N_VAL_PROBS + N_TEST_PROBS:
    n_train = int(total * 0.77)
    n_val   = int(total * 0.115)
    n_test  = total - n_train - n_val
    print(f"Fewer problems ({total}) — adjusting split to {n_train}/{n_val}/{n_test}")
else:
    n_train, n_val, n_test = N_TRAIN_PROBS, N_VAL_PROBS, N_TEST_PROBS

train_probs = problem_ids[:n_train]
val_probs   = problem_ids[n_train:n_train+n_val]
test_probs  = problem_ids[n_train+n_val:n_train+n_val+n_test]

def gather(probs):
    files = []
    for p in probs:
        fs = prob_to_files[p]
        random.shuffle(fs)
        files.extend(fs[:MAX_PER_PROBLEM])
    return files

train_files = gather(train_probs)
val_files   = gather(val_probs)
test_files  = gather(test_probs)

print(f"Train : {len(train_probs)} problems → {len(train_files)} programs")
print(f"Val   : {len(val_probs)} problems → {len(val_files)} programs")
print(f"Test  : {len(test_probs)} problems → {len(test_files)} programs")

## 4 · Compile All Programs to IR

In [ ]:
from tqdm import tqdm

IR_DIR = Path('/kaggle/working/ir')
for split in ('train','val','test'):
    (IR_DIR/split).mkdir(parents=True,exist_ok=True)

def compile_split(files, split_name):
    out_dir  = IR_DIR / split_name
    ok, fail = 0, 0
    results  = {}  # src_name → ll_path
    for src in tqdm(files, desc=f'Compiling {split_name}'):
        stem = src.stem
        ll   = out_dir / f'{stem}.ll'
        r = subprocess.run(
            ['clang','-S','-emit-llvm','-O0',
             '-Xclang','-disable-O0-optnone',
             '-fno-discard-value-names','-w',
             str(src),'-o',str(ll)],
            capture_output=True, timeout=30
        )
        if r.returncode==0 and ll.exists():
            results[stem] = ll; ok+=1
        else:
            fail+=1
    print(f"  {split_name}: {ok} compiled, {fail} failed")
    return results

train_ll = compile_split(train_files, 'train')
val_ll   = compile_split(val_files,   'val')
test_ll  = compile_split(test_files,  'test')

## 5 · Build and Save Graphs

In [ ]:
GRAPH_DIR = Path('/kaggle/working/graphs')
for split in ('train','val','test'):
    (GRAPH_DIR/split).mkdir(parents=True,exist_ok=True)

# Also record baseline instruction counts (used for reward normalisation)
import json
baseline_counts = {}

def build_split(ll_dict, split_name):
    out_dir = GRAPH_DIR / split_name
    ok, fail = 0, 0
    for stem, ll_path in tqdm(ll_dict.items(), desc=f'Graphs {split_name}'):
        try:
            ir_text = ll_path.read_text(errors='replace')
            g       = ir_to_graph(ir_text)
            n_instr = count_instructions(str(ll_path))
            torch.save({'graph': g, 'baseline_instr': n_instr,
                        'split': split_name, 'name': stem},
                       out_dir / f'{stem}.pt')
            baseline_counts[stem] = n_instr; ok += 1
        except Exception as e:
            fail += 1
    print(f"  {split_name}: {ok} graphs, {fail} failures")

build_split(train_ll, 'train')
build_split(val_ll,   'val')
build_split(test_ll,  'test')

# Save metadata
meta = {
    'n_train': len(list((GRAPH_DIR/'train').glob('*.pt'))),
    'n_val':   len(list((GRAPH_DIR/'val').glob('*.pt'))),
    'n_test':  len(list((GRAPH_DIR/'test').glob('*.pt'))),
    'baseline_counts': baseline_counts,
    'num_opcodes': NUM_OPCODES,
    'random_seed': RANDOM_SEED,
}
with open('/kaggle/working/graphs/metadata.json','w') as f:
    json.dump(meta, f, indent=2)

print("\nSummary:")
for k,v in meta.items():
    if k != 'baseline_counts':
        print(f"  {k}: {v}")

## 6 · Validate Output + Quick Stats

In [ ]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Load a few graphs and verify structure
sample_graphs = list((GRAPH_DIR/'train').glob('*.pt'))[:5]
stats = []
for pt in sample_graphs:
    saved = torch.load(pt)
    g     = saved['graph']
    stats.append({
        'name'    : pt.stem,
        'funcs'   : g['func'].x.shape[0],
        'bbs'     : g['bb'].x.shape[0],
        'instrs'  : g['instr'].x.shape[0],
        'cfg_e'   : g['bb','cfg','bb'].edge_index.shape[1],
        'dfg_e'   : g['instr','dfg','instr'].edge_index.shape[1],
        'baseline': saved['baseline_instr'],
    })

df = pd.DataFrame(stats)
print(df.to_string())
print(f"\n✓ All .pt files have correct HeteroData structure")

# Distribution of instruction counts
all_baselines = list(baseline_counts.values())
plt.figure(figsize=(8,4))
plt.hist(all_baselines, bins=40, color='#4C78A8', edgecolor='white', linewidth=0.4)
plt.xlabel('Instruction count (-O0)')
plt.ylabel('Programs')
plt.title('Distribution of baseline instruction counts')
plt.tight_layout()
plt.savefig('/kaggle/working/graphs/instr_distribution.png', dpi=120)
print(f"\nInstruction counts: min={min(all_baselines)}, median={int(np.median(all_baselines))}, max={max(all_baselines)}")

## 7 · Done — Create Kaggle Dataset

**Output files are in `/kaggle/working/graphs/`.**

To make them available to the training notebooks:
1. Click **Output** (right sidebar) in Kaggle
2. Click **New Dataset**
3. Name it exactly: `compiler-opt-graphs`
4. Click **Create**

Then in the training notebooks, add this dataset under *Data → Add data*.

The dataset will appear at `/kaggle/input/compiler-opt-graphs/`.

In [ ]:
import os
total_graphs = sum(len(list((GRAPH_DIR/s).glob('*.pt'))) for s in ('train','val','test'))
total_mb     = sum(f.stat().st_size for f in GRAPH_DIR.rglob('*.pt')) / 1e6

print(f"Total graphs : {total_graphs}")
print(f"Total size   : {total_mb:.1f} MB")
print(f"Output dir   : /kaggle/working/graphs/")
print()
print("Next steps:")
print("  1. Save as Kaggle dataset: 'compiler-opt-graphs'")
print("  2. Open Kaggle_02_train.ipynb (GPU session)")
print("  3. Open Kaggle_03_ablations.ipynb (GPU session)")
print("  4. Open Kaggle_04_baselines_eval.ipynb (GPU or CPU)")